# Oil-Price Market Analysis

This notebook creates the principal figures used to study crude-oil prices and their transmission into consumer fuel costs and individual equities. It covers WTI and Brent over rolling ten-year windows, highlights periods when Brent traded above $100 per barrel, compares Domino’s share price with Brent, and places U.S. gasoline prices beside crude.

Yahoo Finance supplies futures and equity prices. The Federal Reserve Bank of St. Louis supplies Brent spot and U.S. regular gasoline series.


## 1. Environment and paths

Run this cell first. Generated figures are saved in the project’s `outputs/` directory.


In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf


def find_project_root(start: Path) -> Path:
    """Locate the oil-price project from the repository root or notebook folder."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if candidate.name == "oil-price-article":
            return candidate
        nested = candidate / "oil-price-article"
        if nested.is_dir():
            return nested
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Charts save to: {OUTPUT_DIR}")


## 2. WTI crude over ten years

Download front-month WTI futures and establish the broader oil-price backdrop.


In [ ]:
# Automatically calculate the previous 10 years
end_date = pd.Timestamp.today().normalize()
start_date = end_date - pd.DateOffset(years=10)

# Download WTI crude oil front-month futures data
oil_data = yf.download(
    "CL=F",
    start=start_date,
    end=end_date + pd.Timedelta(days=1),
    auto_adjust=False,
    progress=False
)

# Extract closing prices
oil_prices = oil_data["Close"].squeeze().dropna()

oil_prices.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(
    oil_prices.index,
    oil_prices,
    color="#1F4E79",
    linewidth=1.8
)

ax.axhline(
    y=100,
    color="#C00000",
    linestyle="--",
    linewidth=1.5,
    label="$100 per barrel"
)

ax.set_title(
    "WTI Crude Oil Prices Over the Past 10 Years",
    fontsize=18,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("WTI Price (US Dollars per Barrel)", fontsize=12)

# Show one label for each year
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.35
)

ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "wti_crude_oil_10_years.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

## 3. Brent and the $100 threshold

Highlight the periods in which Brent traded above $100 per barrel.


In [ ]:
end_date = pd.Timestamp.today().normalize()
start_date = end_date - pd.DateOffset(years=10)

# BZ=F is the Yahoo Finance ticker for Brent crude oil futures
brent_data = yf.download(
    "BZ=F",
    start=start_date,
    end=end_date + pd.Timedelta(days=1),
    auto_adjust=False,
    progress=False
)

brent_prices = brent_data["Close"].squeeze().dropna()

brent_prices.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

# Plot Brent prices
ax.plot(
    brent_prices.index,
    brent_prices,
    color="#1F4E79",
    linewidth=1.8,
    label="Brent crude"
)

# Shade areas where Brent is above $100
ax.fill_between(
    brent_prices.index,
    100,
    brent_prices.values,
    where=(brent_prices.values > 100),
    interpolate=True,
    color="#E74C3C",
    alpha=0.35,
    label="Price above $100"
)

# Add the $100 threshold
ax.axhline(
    y=100,
    color="#C00000",
    linestyle="--",
    linewidth=1.5,
    label="$100 per barrel"
)

ax.set_title(
    "Brent Crude Oil Prices Over the Past 10 Years",
    fontsize=18,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Brent Crude Price (US Dollars per Barrel)", fontsize=12)

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.legend(frameon=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Save the chart as a high-resolution PNG
fig.savefig(
    OUTPUT_DIR / "brent_crude_above_100.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Chart saved as brent_crude_above_100.png")

## 4. Corporate sensitivity case study

Compare Domino’s adjusted share price with Brent crude. The chart is descriptive and does not imply that oil alone caused movements in the company’s valuation.


In [ ]:
end_date = pd.Timestamp.today().normalize()
start_date = end_date - pd.DateOffset(years=10)

# Download adjusted Domino's stock prices
dpz_data = yf.download(
    "DPZ",
    start=start_date,
    end=end_date + pd.Timedelta(days=1),
    auto_adjust=True,
    progress=False
)

# Download Brent crude oil futures prices
brent_data = yf.download(
    "BZ=F",
    start=start_date,
    end=end_date + pd.Timedelta(days=1),
    auto_adjust=False,
    progress=False
)

dpz_prices = dpz_data["Close"].squeeze().dropna()
brent_prices = brent_data["Close"].squeeze().dropna()

print("Domino's data:")
display(dpz_prices.head())

print("Brent data:")
display(brent_prices.head())

In [ ]:
fig, ax1 = plt.subplots(figsize=(15, 8))

# Domino's Pizza stock price — left axis
dpz_line = ax1.plot(
    dpz_prices.index,
    dpz_prices,
    color="#00539F",
    linewidth=2,
    label="Domino's Pizza stock price"
)

ax1.set_xlabel("Year", fontsize=12)
ax1.set_ylabel(
    "Domino's Pizza Share Price (US$)",
    color="#00539F",
    fontsize=12
)
ax1.tick_params(axis="y", labelcolor="#00539F")

# Brent crude price — right axis
ax2 = ax1.twinx()

brent_line = ax2.plot(
    brent_prices.index,
    brent_prices,
    color="#D35400",
    linewidth=1.8,
    label="Brent crude oil"
)

ax2.set_ylabel(
    "Brent Crude Price (US$ per Barrel)",
    color="#D35400",
    fontsize=12
)
ax2.tick_params(axis="y", labelcolor="#D35400")

# Highlight areas where Brent is above $100
ax2.fill_between(
    brent_prices.index,
    100,
    brent_prices.values,
    where=(brent_prices.values > 100),
    interpolate=True,
    color="#E74C3C",
    alpha=0.25,
    label="Brent above $100"
)

# Add $100 threshold
threshold_line = ax2.axhline(
    y=100,
    color="#C00000",
    linestyle="--",
    linewidth=1.3,
    label="$100 oil threshold"
)

ax1.set_title(
    "Domino's Pizza Stock Price and Brent Crude Oil Prices",
    fontsize=18,
    fontweight="bold",
    pad=15
)

ax1.text(
    0.5,
    1.01,
    "Ten-year comparison, with periods of Brent above $100 highlighted",
    transform=ax1.transAxes,
    ha="center",
    fontsize=11,
    color="#555555"
)

# Format dates
ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# Combine legends from both axes
lines = dpz_line + brent_line + [threshold_line]
labels = [line.get_label() for line in lines]

# Add shaded-region legend entry
handles2, labels2 = ax2.get_legend_handles_labels()
if "Brent above $100" in labels2:
    position = labels2.index("Brent above $100")
    lines.append(handles2[position])
    labels.append(labels2[position])

ax1.legend(
    lines,
    labels,
    loc="upper left",
    frameon=False
)

ax1.grid(axis="both", linestyle="--", alpha=0.25)
ax1.spines["top"].set_visible(False)
ax2.spines["top"].set_visible(False)

plt.tight_layout()

# Save a high-resolution copy
fig.savefig(
    OUTPUT_DIR / "dominos_stock_vs_brent_oil_10_years.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Chart saved as dominos_stock_vs_brent_oil_10_years.png")

## 5. Brent and U.S. gasoline prices

Compare weekly Brent prices with U.S. regular gasoline prices to illustrate downstream pass-through.


In [ ]:
# Set a rolling ten-year period
end_date = pd.Timestamp.today().normalize()
start_date = end_date - pd.DateOffset(years=10)

start_text = start_date.strftime("%Y-%m-%d")
end_text = end_date.strftime("%Y-%m-%d")

# FRED download URLs
brent_url = (
    "https://fred.stlouisfed.org/graph/fredgraph.csv"
    f"?id=DCOILBRENTEU&cosd={start_text}&coed={end_text}"
)

gas_url = (
    "https://fred.stlouisfed.org/graph/fredgraph.csv"
    f"?id=GASREGW&cosd={start_text}&coed={end_text}"
)

# Download without assuming the date-column name
brent = pd.read_csv(brent_url)
gas = pd.read_csv(gas_url)

# Rename the first column to "date"
# This works whether FRED calls it DATE or observation_date
brent = brent.rename(
    columns={
        brent.columns[0]: "date",
        "DCOILBRENTEU": "brent_price"
    }
)

gas = gas.rename(
    columns={
        gas.columns[0]: "date",
        "GASREGW": "gas_price"
    }
)

# Convert the date columns
brent["date"] = pd.to_datetime(brent["date"])
gas["date"] = pd.to_datetime(gas["date"])

# Convert price columns to numeric
brent["brent_price"] = pd.to_numeric(
    brent["brent_price"],
    errors="coerce"
)

gas["gas_price"] = pd.to_numeric(
    gas["gas_price"],
    errors="coerce"
)

# Remove missing observations
brent = brent.dropna(subset=["brent_price"])
gas = gas.dropna(subset=["gas_price"])

# Convert daily Brent prices into weekly averages ending Monday
brent_weekly = (
    brent.set_index("date")["brent_price"]
    .resample("W-MON")
    .mean()
    .reset_index()
)

# Merge Brent and gasoline observations
combined = pd.merge(
    brent_weekly,
    gas,
    on="date",
    how="inner"
)

display(combined.head())
print(f"Number of weekly observations: {len(combined)}")

In [ ]:
fig, ax1 = plt.subplots(figsize=(15, 8))

# Brent crude — left axis
brent_line = ax1.plot(
    combined["date"],
    combined["brent_price"],
    color="#1F4E79",
    linewidth=2,
    label="Brent crude oil"
)

ax1.set_xlabel("Year", fontsize=12)
ax1.set_ylabel(
    "Brent Crude Price (US$ per Barrel)",
    color="#1F4E79",
    fontsize=12
)

ax1.tick_params(axis="y", labelcolor="#1F4E79")

# Highlight periods when Brent exceeded $100
above_100 = ax1.fill_between(
    combined["date"],
    100,
    combined["brent_price"],
    where=(combined["brent_price"] > 100),
    interpolate=True,
    color="#E74C3C",
    alpha=0.22,
    label="Brent above $100"
)

threshold_line = ax1.axhline(
    100,
    color="#C00000",
    linestyle="--",
    linewidth=1.2,
    label="$100 Brent threshold"
)

# Gasoline — right axis
ax2 = ax1.twinx()

gas_line = ax2.plot(
    combined["date"],
    combined["gas_price"],
    color="#F2A900",
    linewidth=2,
    label="U.S. regular gasoline"
)

ax2.set_ylabel(
    "Regular Gasoline Price (US$ per Gallon)",
    color="#B57600",
    fontsize=12
)

ax2.tick_params(axis="y", labelcolor="#B57600")

# Titles
ax1.set_title(
    "Gasoline Prices Follow the Crude-Oil Chain",
    fontsize=18,
    fontweight="bold",
    pad=28
)

ax1.text(
    0.5,
    1.015,
    "Weekly U.S. gasoline prices and Brent crude oil prices over the past 10 years",
    transform=ax1.transAxes,
    ha="center",
    fontsize=11,
    color="#555555"
)

# Format x-axis
ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# Combine legends
legend_items = brent_line + gas_line + [threshold_line, above_100]
legend_labels = [item.get_label() for item in legend_items]

ax1.legend(
    legend_items,
    legend_labels,
    loc="upper left",
    frameon=False
)

# Styling
ax1.grid(axis="y", linestyle="--", alpha=0.3)
ax1.spines["top"].set_visible(False)
ax2.spines["top"].set_visible(False)

plt.tight_layout()

# Save before displaying
fig.savefig(
    OUTPUT_DIR / "gasoline_vs_brent_crude_10_years.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Chart saved as gasoline_vs_brent_crude_10_years.png")

## Limitations

Front-month futures can be affected by contract rolling and do not equal physical-market transaction prices. The Domino’s comparison is illustrative rather than causal, while dual-axis charts can visually exaggerate relationships. Data are downloaded through the current date, so future reruns may differ from the committed figures.
